# 08 — Final Model Selection

## 1. Objective
The goal of this notebook is to execute the **formal final model selection** for the Disease Diagnosis Prediction project, comparing **Tuned Logistic Regression** and **Tuned Support Vector Machine** across the complete evidence framework established in Phases 4 through 7.

### Decision Framework & Rules:
1. **Multi-Criteria Hierarchy:** Decision is based on a structured hierarchy of primary metrics (ROC-AUC, F1, Recall, Specificity, FPR/FNR), secondary metrics (CV stability, AP, Brier score, threshold behavior), and practical considerations (interpretability and complexity).
2. **No Model Persistence:** `models/` directory remains completely empty during Phase 8.
3. **No Clinical Claims:** The selected model is identified as the final machine-learning model for this project demonstration and is NOT medically validated to diagnose disease.


## 2. Load Project Modules & Candidate Model Evidence
Import project selection modules and load candidate metrics.


In [1]:
import sys
from pathlib import Path

# Locate project root containing src/
root_dir = Path.cwd()
for p in [root_dir] + list(root_dir.parents):
    if (p / 'src' / 'config.py').exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break

import pandas as pd
import numpy as np
from IPython.display import display

from src.config import REPORTS_DIR, FIGURES_DIR
from src.model_selection import (
    build_final_model_selection_df,
    generate_final_model_selection_summary,
    run_final_model_selection,
)

# Execute final model selection pipeline
selection_df, selected_model_name = run_final_model_selection()

print(f"SELECTED FINAL MODEL: {selected_model_name}\n")
display(selection_df)


C:\projects\Disease-Diagnosis-Prediction\src\model_selection.py:166: SyntaxWarning: invalid escape sequence '\%'
  """


SELECTED FINAL MODEL: Tuned Support Vector Machine



,Model,CV_ROC_AUC_Mean,CV_ROC_AUC_Std,Test_ROC_AUC,Test_Accuracy,Test_Precision,Test_Recall,Test_Specificity,Test_F1,Test_FPR,Test_FNR,Average_Precision,Brier_Score,Interpretability,Selection_Status
0,Tuned Logistic Regression,0.879513,0.046341,0.927308,0.826087,0.830189,0.862745,0.780488,0.846154,0.219512,0.137255,0.945282,0.137263,High (Direct Log-Odds Coefficients),Rejected
1,Tuned Support Vector Machine,0.879872,0.044217,0.929460,0.847826,0.855769,0.872549,0.817073,0.864078,0.182927,0.127451,0.945498,0.137632,Moderate-High (Linear Kernel Weights),Selected


## 3. Quantitative Performance Comparison
Compare training 5-fold cross-validation performance and held-out test set metrics for both candidates.


In [2]:
print("=== Primary Performance Metrics ===")
display(selection_df[["Model", "CV_ROC_AUC_Mean", "CV_ROC_AUC_Std", "Test_ROC_AUC", "Test_Accuracy", "Test_F1", "Selection_Status"]])


=== Primary Performance Metrics ===


,Model,CV_ROC_AUC_Mean,CV_ROC_AUC_Std,Test_ROC_AUC,Test_Accuracy,Test_F1,Selection_Status
0,Tuned Logistic Regression,0.879513,0.046341,0.927308,0.826087,0.846154,Rejected
1,Tuned Support Vector Machine,0.879872,0.044217,0.929460,0.847826,0.864078,Selected


## 4. Error Profile & Diagnostic Trade-offs
Compare false positive rates (FPR), false negative rates (FNR), Sensitivity, and Specificity across candidate models on the held-out test set ($N=184$).


In [3]:
print("=== Error Profile Metrics ===")
display(selection_df[["Model", "Test_Recall", "Test_Specificity", "Test_FPR", "Test_FNR", "Test_Precision"]])


=== Error Profile Metrics ===


,Model,Test_Recall,Test_Specificity,Test_FPR,Test_FNR,Test_Precision
0,Tuned Logistic Regression,0.862745,0.780488,0.219512,0.137255,0.830189
1,Tuned Support Vector Machine,0.872549,0.817073,0.182927,0.127451,0.855769


## 5. Probability Calibration & Threshold Considerations
Inspect Out-of-Fold Average Precision and Brier scores evaluated on training data.


In [4]:
print("=== Probability & Ranking Metrics ===")
display(selection_df[["Model", "Average_Precision", "Brier_Score"]])


=== Probability & Ranking Metrics ===


,Model,Average_Precision,Brier_Score
0,Tuned Logistic Regression,0.945282,0.137263
1,Tuned Support Vector Machine,0.945498,0.137632


## 6. Interpretability & Model Complexity Considerations
Evaluate structural model complexity and feature weight inspectability.

- **Logistic Regression:** Direct log-odds coefficients available. All categorical levels one-hot encoded without dropping.
- **Support Vector Machine:** `kernel='linear'` chosen via hyperparameter tuning. Linear feature weights (`coef_`) provide feature importance inspectability, though margin-based decision boundaries differ from log-odds probability ratios.


In [5]:
print("=== Interpretability Summary ===")
display(selection_df[["Model", "Interpretability", "Selection_Status"]])


=== Interpretability Summary ===


,Model,Interpretability,Selection_Status
0,Tuned Logistic Regression,High (Direct Log-Odds Coefficients),Rejected
1,Tuned Support Vector Machine,Moderate-High (Linear Kernel Weights),Selected


## 7. Final Model Decision & Rationale

### Selected Final Model: **Tuned Support Vector Machine** (`C=100, kernel='linear', gamma='scale', probability=True`)
### Rejected Alternative: **Tuned Logistic Regression** (`C=0.1, solver='lbfgs', max_iter=1000`)

### Rationale:
1. **Multi-Metric Superiority:** Outperforms Tuned Logistic Regression across all primary test metrics, including Accuracy (84.78%), F1 (0.8641), Recall (87.25%), Specificity (81.71%), and Test ROC-AUC (0.9295).
2. **Balanced Misclassifications:** Lowers both false positive errors (15 vs. 18) and false negative errors (13 vs. 14).
3. **Cross-Validation Stability:** Achieves highest 5-fold CV mean ROC-AUC (0.8799 ± 0.0442).
4. **Low Structural Complexity:** Maintains a linear decision boundary (`kernel='linear'`) with direct coefficient inspectability.


## 8. Limitations & Methodological Scope

> **CAUTIOUS METHODOLOGICAL SCOPE & NON-CLINICAL NOTICE:**
> 1. **Machine-Learning Project Scope:** Tuned Support Vector Machine is the selected final machine-learning model for this project.
> 2. **Non-Clinical Application:** Statistical metrics evaluated on tabular observations do NOT establish medical diagnostic accuracy, efficacy, safety, or clinical utility. The model is NOT medically validated and cannot diagnose disease.
> 3. **Non-Causal Associations:** Model parameters and feature weights represent predictive associations within the transformed feature space and do NOT imply causal biological relationships.
> 4. **No Saved Binary in Phase 8:** The `models/` directory remains empty until Phase 9 persistence.
